<a href="https://colab.research.google.com/github/yaesur/business_python/blob/main/%EC%9E%AC%EA%B3%B5%EA%B8%89_%EC%8B%A0%EA%B7%9C%EA%B3%B5%EA%B8%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from scipy import stats

file_name = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file_name)
sheet_names = xls.sheet_names
cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]

all_data = []

for i, name in enumerate(sheet_names):

  df = pd.read_excel(file_name, sheet_name=name, skiprows=2, header=None)

  temp = pd.DataFrame()
  temp['공급유형'] = df.iloc[:, 0].astype(str).str.strip()
  temp['점수'] = df.iloc[:, cols[i]]

  all_data.append(temp)

final_df = pd.concat(all_data).dropna(subset=['공급유형', '점수'])

final_df['점수'] = pd.to_numeric(final_df['점수'].astype(str).str.replace('점', ''), errors='coerce')
final_df = final_df.dropna(subset=['점수'])

final_df = final_df[final_df['공급유형'].str.contains('신규|재공급')]

summary = final_df.groupby('공급유형')['점수'].agg(['count', 'mean', 'std']).round(2)
summary.columns = ['공급건수', '평균점수', '표준편차']

print("=== 신규 vs 재공급 분석 결과 ===")
print(summary)

# T-test로 유의성 검정
new_group = final_df[final_df['공급유형'].str.contains('신규')]['점수']
re_group = final_df[final_df['공급유형'].str.contains('재공급')]['점수']

t_stat, p_val = stats.ttest_ind(new_group, re_group)

print(f"\n[T-test 결과] p-value: {p_val:.10f}")

if p_val < 0.05:
    print("결론: 신규/재공급 여부는 합격 예측에 '매우 중요한' 변수입니다.")
else:
    print("결론: 두 유형 간 점수 차이가 크지 않아 변수에서 제외해도 무방합니다.")

=== 신규 vs 재공급 분석 결과 ===
      공급건수  평균점수  표준편차
공급유형                  
신규공급   542  5.81  2.14
재공급    580  6.19  2.15

[T-test 결과] p-value: 0.0027580827
결론: 신규/재공급 여부는 합격 예측에 '매우 중요한' 변수입니다.


In [ ]:
import pandas as pd
from scipy import stats

file_name = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file_name)
sheet_names = xls.sheet_names

# [순위] 컬럼과 [점수] 컬럼 인덱스 설정
level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]

all_data = []

for i, name in enumerate(sheet_names):
    df = pd.read_excel(file_name, sheet_name=name, skiprows=2, header=None)

    temp = pd.DataFrame()
    temp['공급유형'] = df.iloc[:, 0].astype(str).str.strip()
    temp['순위'] = df.iloc[:, level_cols[i]]
    temp['점수'] = df.iloc[:, score_cols[i]]

    all_data.append(temp)

final_df = pd.concat(all_data).dropna(subset=['공급유형', '순위', '점수'])

# 순위 데이터 정제 (숫자만 남기기)
final_df['순위'] = final_df['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final_df['순위'] = pd.to_numeric(final_df['순위'], errors='coerce')

# 점수 데이터 정제
final_df['점수'] = pd.to_numeric(final_df['점수'].astype(str).str.replace('점', '', regex=False).str.strip(), errors='coerce')

# 결측치 최종 제거
final_df = final_df.dropna(subset=['순위', '점수'])

# '신규' 또는 '재공급'이라는 단어가 포함된 행만 필터링
final_df = final_df[final_df['공급유형'].str.contains('신규|재공급')]

# 공급유형 텍스트 깔끔하게 통일 (예: '신규공급', '신규주택' -> '신규'로 통일)
final_df.loc[final_df['공급유형'].str.contains('신규'), '공급유형_그룹'] = '신규'
final_df.loc[final_df['공급유형'].str.contains('재공급'), '공급유형_그룹'] = '재공급'


# ========================================================
# 🎯 [순위별 분리 리설계] 1순위 / 2순위 각각 T-검정 실시
# ========================================================

for rank in [1, 2, 3]:
    print(f"\n==================================================")
    print(f"       🎯 [{rank}순위 모집 단위] 신규 vs 재공급 분석       ")
    print(f"==================================================")

    # 해당 순위 데이터만 필터링
    rank_df = final_df[final_df['순위'] == rank]

    if rank_df.empty:
        print(f"❌ {rank}순위 커트라인 데이터가 존재하지 않습니다.")
        continue

    # 1. 공급유형 그룹별 통계량 계산
    summary = rank_df.groupby('공급유형_그룹')['점수'].agg(['count', 'mean', 'std']).round(2)
    summary.columns = ['공급건수', '평균점수', '표준편차']

    print(f"📊 [{rank}순위] 기초 통계량")
    print(summary)
    print("-" * 50)

    # 2. T-test를 위한 집단 분리
    new_group = rank_df[rank_df['공급유형_그룹'] == '신규']['점수']
    re_group = rank_df[rank_df['공급유형_그룹'] == '재공급']['점수']

    # T-test 조건 체크 (두 집단 모두 데이터가 최소 2개 이상 있어야 함)
    if len(new_group) > 1 and len(re_group) > 1:
        t_stat, p_val = stats.ttest_ind(new_group, re_group)
        print(f"📈 [T-test 결과] p-value: {p_val:.10f}")

        if p_val < 0.05:
            print(f">>> 결론: {rank}순위에서 신규/재공급 여부는 합격 예측에 '매우 중요한' 변수입니다.")
            print(f"    (두 유형 간 평균 점수 차이가 통계적으로 유의미합니다.)")
        else:
            print(f">>> 결론: {rank}순위에서 두 유형 간 점수 차이가 우연일 수 있습니다.")
            print(f"    (예측 모델 변수 선택 시 후순위로 고려해도 무방합니다.)")
    else:
        print("⚠ 신규 또는 재공급 데이터 부족으로 T-test를 수행할 수 없습니다.")

    print("==================================================")


       🎯 [1순위 모집 단위] 신규 vs 재공급 분석       
📊 [1순위] 기초 통계량
         공급건수  평균점수  표준편차
공급유형_그룹                  
신규        320  5.63  2.39
재공급       349  6.21  2.39
--------------------------------------------------
📈 [T-test 결과] p-value: 0.0018015605
>>> 결론: 1순위에서 신규/재공급 여부는 합격 예측에 '매우 중요한' 변수입니다.
    (두 유형 간 평균 점수 차이가 통계적으로 유의미합니다.)

       🎯 [2순위 모집 단위] 신규 vs 재공급 분석       
📊 [2순위] 기초 통계량
         공급건수  평균점수  표준편차
공급유형_그룹                  
신규        210  6.10  1.72
재공급       223  6.17  1.74
--------------------------------------------------
📈 [T-test 결과] p-value: 0.6728804056
>>> 결론: 2순위에서 두 유형 간 점수 차이가 우연일 수 있습니다.
    (예측 모델 변수 선택 시 후순위로 고려해도 무방합니다.)

       🎯 [3순위 모집 단위] 신규 vs 재공급 분석       
📊 [3순위] 기초 통계량
         공급건수  평균점수  표준편차
공급유형_그룹                  
신규         12  5.25  0.97
재공급         8  5.75  1.28
--------------------------------------------------
📈 [T-test 결과] p-value: 0.3321940140
>>> 결론: 3순위에서 두 유형 간 점수 차이가 우연일 수 있습니다.
    (예측 모델 변수 선택 시 후순위로 고려해도 무방합니다.)
